# 5-Year Pipeline: Part 1 - Download Data

In [ ]:
# --- STEP 1: DOWNLOAD 5 YEARS OF RAW DATA ---
import os
import io
import datetime
import time
import requests
import pandas as pd
import yfinance as yf

def get_nifty200_symbols():
    """Fetches the official NIFTY 200 constituent tickers directly from NIFTY Indices."""
    url = "https://www.niftyindices.com/IndexConstituent/ind_nifty200list.csv"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        df_csv = pd.read_csv(io.StringIO(response.text))
        symbols = [f"{sym}.NS" for sym in df_csv['Symbol'].dropna().unique()]
        return symbols
    except Exception as e:
        print(f"Failed to fetch live Nifty 200 list ({e}), using fallback list.")
        return ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS"]

def get_sector_index(sector):
    """Maps Yahoo Finance sectors to approximate NIFTY Sector Indices."""
    mapping = {
        "Financial Services": "NIFTY FINANCIAL SERVICES",
        "Information Technology": "NIFTY IT",
        "Healthcare": "NIFTY PHARMA",
        "Consumer Defensive": "NIFTY FMCG",
        "Consumer Cyclical": "NIFTY AUTO",  # Auto is a major part of consumer cyclical in India
        "Basic Materials": "NIFTY METAL",
        "Energy": "NIFTY ENERGY",
        "Industrials": "NIFTY INFRA",
        "Communication Services": "NIFTY MEDIA",
        "Utilities": "NIFTY ENERGY",
        "Real Estate": "NIFTY REALTY"
    }
    return mapping.get(sector, "Unknown")

def download_data():
    symbols = get_nifty200_symbols()
    
    # Create output directory
    output_dir = "5_year_data/raw"
    os.makedirs(output_dir, exist_ok=True)
    
    # We need last 1250 working days (approx 2.25 years of calendar days)
    # So downloading the last 1850 days ensures we have enough data points.
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=1850)
    end_date = today # end is exclusive in yfinance, but we'll also filter manually to be safe
    
    print("Downloading NIFTY 50 data...")
    nifty50 = yf.download("^NSEI", start=start_date, end=end_date, auto_adjust=False, progress=False)
    
    # Handle multi-index columns from yfinance batch downloads
    if isinstance(nifty50.columns, pd.MultiIndex):
        nifty50.columns = nifty50.columns.droplevel(1)
        
    # Ensure today's date is not included
    nifty50 = nifty50[nifty50.index.date < today]
    
    # Take the last 1250 days
    nifty50 = nifty50.tail(1250)
    
    nifty_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    # If Volume is not present in Nifty50, we add it with 0s
    for c in nifty_cols:
        if c not in nifty50.columns:
            nifty50[c] = 0
            
    nifty50 = nifty50[nifty_cols]
    nifty50.rename(columns={
        'Open': 'Nifty50_Open',
        'High': 'Nifty50_High',
        'Low': 'Nifty50_Low',
        'Close': 'Nifty50_Close',
        'Volume': 'Nifty50_Volume'
    }, inplace=True)
    
    total = len(symbols)
    
    for i, symbol in enumerate(symbols, 1):
        clean_symbol = symbol.replace('.NS', '')
        print(f"[{i}/{total}] Processing {clean_symbol}...")
        
        try:
            # Download stock data individually to save memory
            df = yf.download(symbol, start=start_date, end=end_date, auto_adjust=False, progress=False)
            
            if df.empty:
                print(f"  No data for {symbol}")
                continue
                
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)
                
            # Filter out today's data strictly
            df = df[df.index.date < today]
            
            # Take the last 1250 rows
            df = df.tail(1250)
            
            # Fetch Sector info from yfinance Ticker (lightweight call)
            # Using try-except because info can sometimes fail for specific tickers
            ticker_obj = yf.Ticker(symbol)
            try:
                info = ticker_obj.info
                sector = info.get('sector', 'Unknown')
            except Exception:
                sector = 'Unknown'
            
            # Prepare dataframe
            df_final = df.copy()
            df_final['Stock Symbol'] = clean_symbol
            df_final['Sector'] = sector
            df_final['Sector Index'] = get_sector_index(sector)
            
            # Trading/market calendar component (Day of the week)
            df_final['Trading_Day_Of_Week'] = df_final.index.day_name()
            
            # Ensure 'Adj Close' exists (older / newer yf compatibility)
            if 'Adj Close' not in df_final.columns:
                if 'Close' in df_final.columns:
                    df_final['Adj Close'] = df_final['Close']
                else:
                    df_final['Adj Close'] = None

            # Join Nifty50 columns
            df_final = df_final.join(nifty50, how='left')
            
            cols_needed = [
                'Stock Symbol', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 
                'Sector', 'Sector Index', 
                'Nifty50_Open', 'Nifty50_High', 'Nifty50_Low', 'Nifty50_Close', 'Nifty50_Volume',
                'Trading_Day_Of_Week'
            ]
            
            # Add any missing columns as None to ensure uniform format
            for c in cols_needed:
                if c not in df_final.columns:
                    df_final[c] = None
                    
            # Reorder columns
            df_final = df_final[cols_needed]
            
            # Save to CSV in output directory
            output_file = os.path.join(output_dir, f"{clean_symbol}_data.csv")
            if os.path.exists(output_file):
                print(f"  {clean_symbol} already exists, skipping...")
                continue
            df_final.to_csv(output_file, index_label='Date')
            
            # Memory efficiency: immediately delete DataFrames from memory
            del df
            del df_final
            
        except Exception as e:
            print(f"  Error processing {symbol}: {e}")
            
        # Small delay to prevent being rate limited by Yahoo Finance
        time.sleep(0.1)
        
    print(f"\nAll downloads completed successfully! Check the '{output_dir}' directory.")

if __name__ == "__main__":
    download_data()

# UNCOMMENT NEXT LINE TO RUN DOWNLOAD
download_data()
